In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

# Add the main project folder to Python's path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
    
print(f"Project root added to path: {project_root}")

Project root added to path: /home/ikr/Documents/gitrepos/energivejr/nowcastingvalidation


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

# Import the classes you built!
from validator import SatelliteNowcastLoader, SatelliteObservationLoader, ScoresCalculator

# Set a nice style for our plots
plt.style.use('bmh')

In [5]:

start_date = "2026-02-25"
end_date = "2026-02-26"

print(f"Loading data from {start_date} to {end_date}...")

# 1. Load Nowcasts
nwc_loader = SatelliteNowcastLoader()
nowcasts = nwc_loader.load_data(start_date, end_date)

# 2. Load Observations
obs_loader = SatelliteObservationLoader()
observations = obs_loader.load_data(start_date, end_date)

# IMPORTANT DATA CHECK: 
# If your observation variable is named something like 'SIS' instead of 'GHI', 
# uncomment the line below and change 'SIS' to whatever it actually is!
# observations = observations.rename({'SIS': 'GHI'})

print("\n--- Available Variables ---")
print("Nowcasts:", list(nowcasts.data_vars))
print("Observations:", list(observations.data_vars))

Loading data from 2026-02-25 to 2026-02-26...


RuntimeError: Failed to load NetCDF files: No module named 'h5py', backend not available. Please install 'h5py' into your Python environment.

In [ ]:
print("Initializing Evaluator (aligning spatial and temporal coordinates)...")
evaluator = SolarEvaluator(obs_ds=observations, fcst_ds=nowcasts)

print("Calculating metrics...")
# We will test two irradiance thresholds (100 and 300 W/m2) 
# and two neighborhood sizes for the FSS (5x5 and 11x11 pixels)
results = evaluator.run_all_metrics(
    thresholds=[100, 300], 
    window_sizes=[[5, 5], [11, 11]]
)

print("Done! Here is your results dataset:")
results

In [ ]:
import matplotlib.pyplot as plt

# Create a nice, wide figure
fig, ax = plt.subplots(figsize=(12, 5))

# Plot just the RMSE variable from your results dataset
results['RMSE'].plot(
    ax=ax, 
    color='#e74c3c', # A nice distinct red
    linewidth=2,
    label='Nowcast RMSE'
)

# Make it look professional
ax.set_title("Solar GHI Nowcast: Root Mean Square Error (RMSE)", fontsize=14, fontweight='bold')
ax.set_ylabel("Error (W/m²)", fontsize=12)
ax.set_xlabel("Time (UTC)", fontsize=12)

# Add a grid to make it easier to read values
ax.grid(True, linestyle='--', alpha=0.7)
ax.legend(loc='upper right')

# Tight layout prevents labels from getting cut off
plt.tight_layout()
plt.show()